!pip install pymongo mongomock


In [33]:
import mongomock
from pymongo import ASCENDING, DESCENDING  # solo constantes, compatible con mongomock

### Constructor

In [23]:
client = mongomock.MongoClient()

### creo la bbdd "demo" y la colección "alumnos"

In [71]:
db  = client["demo"] ##bbdd
alu = db["alumnos"] #coleccion

### insertamos datos

In [25]:
alu.insert_many([
    {"nombre": "Ana", "nota": 8, "ciclo": "ASIR"},
    {"nombre": "Luis", "nota": 5, "ciclo": "DAW"},
    {"nombre": "Marta", "nota": 9, "ciclo": "DAW"},
    {"nombre": "David", "nota":5, "ciclo": "DAW"},
    {"nombre": "Paola", "nota":9, "ciclo": "DAM"},
])

InsertManyResult([ObjectId('6989c463788be09fc21060e7'), ObjectId('6989c463788be09fc21060e8'), ObjectId('6989c463788be09fc21060e9'), ObjectId('6989c463788be09fc21060ea'), ObjectId('6989c463788be09fc21060eb')], acknowledged=True)

In [76]:
#numero de documentos
alu.count_documents({})

8

### operaciones

In [31]:
#todas las filas
list(alu.find({}))

[{'nombre': 'Ana',
  'nota': 8,
  'ciclo': 'ASIR',
  '_id': ObjectId('6989c463788be09fc21060e7')},
 {'nombre': 'Luis',
  'nota': 5,
  'ciclo': 'DAW',
  '_id': ObjectId('6989c463788be09fc21060e8')},
 {'nombre': 'Marta',
  'nota': 9,
  'ciclo': 'DAW',
  '_id': ObjectId('6989c463788be09fc21060e9')},
 {'nombre': 'David',
  'nota': 5,
  'ciclo': 'DAW',
  '_id': ObjectId('6989c463788be09fc21060ea')},
 {'nombre': 'Paola',
  'nota': 9,
  'ciclo': 'DAM',
  '_id': ObjectId('6989c463788be09fc21060eb')}]

In [26]:
#primera fila
alu.find_one()

{'nombre': 'Ana',
 'nota': 8,
 'ciclo': 'ASIR',
 '_id': ObjectId('6989c463788be09fc21060e7')}

In [27]:
# las 2 primeras filas
cursor = alu.find({}).limit(2)
list(cursor)

[{'nombre': 'Ana',
  'nota': 8,
  'ciclo': 'ASIR',
  '_id': ObjectId('6989c463788be09fc21060e7')},
 {'nombre': 'Luis',
  'nota': 5,
  'ciclo': 'DAW',
  '_id': ObjectId('6989c463788be09fc21060e8')}]

In [28]:
# Mostrar campos (sin _id)
list(alu.find({}, {"nombre": 1, "nota": 1, "ciclo": 1,"_id": 0}).limit(2))

[{'nombre': 'Ana', 'nota': 8, 'ciclo': 'ASIR'},
 {'nombre': 'Luis', 'nota': 5, 'ciclo': 'DAW'}]

In [34]:
# Top 3 alumos por nota (descendente)
list(
    alu.find({}, {"_id": 0, "nombre": 1, "nota": 1})
         .sort("nota", DESCENDING)
         .limit(3)
)

[{'nombre': 'Marta', 'nota': 9},
 {'nombre': 'Paola', 'nota': 9},
 {'nombre': 'Ana', 'nota': 8}]

#### Filtros (WHERE)

Operadores típicos:

$gt, $gte, $lt, $lte (comparaciones)
$in (pertenencia)
$regex (patrones)
$and, $or (lógicos)

In [44]:
# Comparación: nota >= 8
q = {"nota": {"$gte": 8}}
list(alu.find(q, {"_id": 0, "nombre": 1, "nota": 1}).limit(5))

[{'nombre': 'Ana', 'nota': 8},
 {'nombre': 'Marta', 'nota': 9},
 {'nombre': 'Paola', 'nota': 9}]

In [45]:
# Comparación: nota > 8
q = {"nota": {"$gt": 8}}
list(alu.find(q, {"_id": 0, "nombre": 1, "nota": 1}).limit(5))


[{'nombre': 'Marta', 'nota': 9}, {'nombre': 'Paola', 'nota': 9}]

In [40]:
#  $in: categoria en un conjunto
q = {"ciclo": {"$in": ["ASIR", "DAM"]}}
list(alu.find(q,{}))

[{'nombre': 'Ana',
  'nota': 8,
  'ciclo': 'ASIR',
  '_id': ObjectId('6989c463788be09fc21060e7')},
 {'nombre': 'Paola',
  'nota': 9,
  'ciclo': 'DAM',
  '_id': ObjectId('6989c463788be09fc21060eb')}]

In [41]:
# $regex: nombre empieze por 'M' 
q = {"nombre": {"$regex": "^M"}}
list(alu.find(q,{}))

[{'nombre': 'Marta',
  'nota': 9,
  'ciclo': 'DAW',
  '_id': ObjectId('6989c463788be09fc21060e9')}]

In [46]:
# $or: (ciclo='ASIR') OR (nota>9)
q = {"$or": [{"ciclo": "ASIR"}, {"nota": {"$gt": 9}}]}
list(alu.find(q,{}))

[{'nombre': 'Ana',
  'nota': 8,
  'ciclo': 'ASIR',
  '_id': ObjectId('6989c463788be09fc21060e7')}]

In [47]:
# $or: (ciclo='ASIR') AND (nota>9)
q = {"$and": [{"ciclo": "ASIR"}, {"nota": {"$gt": 9}}]}
list(alu.find(q,{}))

[]

### Funciones de grupo

In [63]:
#meda + ordenacion descendente 
pipeline = [
    {"$group": {"_id": "$ciclo", "media": {"$avg": "$nota"}}},
    {"$sort": {"media": -1}}, #descendente
    {"$limit": 5},
]
list(alu.aggregate(pipeline))


[{'media': 9.0, '_id': 'DAM'},
 {'media': 8.0, '_id': 'ASIR'},
 {'media': 6.333333333333333, '_id': 'DAW'}]

In [66]:
#meda + ordenacion ascendente
pipeline = [
        {"$group": {"_id": "$ciclo", "media": {"$avg": "$nota"}}},
        {"$sort": {"media": 1}}, #ascendente
    ]
list(alu.aggregate(pipeline))

[{'media': 6.333333333333333, '_id': 'DAW'},
 {'media': 8.0, '_id': 'ASIR'},
 {'media': 9.0, '_id': 'DAM'}]

In [68]:
#maximo por grupo
pipeline = [
        {"$group": {"_id": "$ciclo", "max": {"$max": "$nota"}}},
        {"$sort": {"max": 1}}, #ascendente
    ]
list(alu.aggregate(pipeline))

[{'max': 8, '_id': 'ASIR'}, {'max': 9, '_id': 'DAM'}, {'max': 9, '_id': 'DAW'}]

In [67]:
#filas donde la nota es la máxima o la mínima
pipeline = [
        {"$facet": {
            "max": [
                {"$sort": {"nota": -1}}, #descendente
                {"$limit": 1},
                {"$project": {"_id": 0, "nombre": 1, "ciclo": 1, "nota": 1}}
            ],
            "min": [
                {"$sort": {"nota": 1}}, #ascendente
                {"$limit": 1},
                {"$project": {"_id": 0, "nombre": 1, "ciclo": 1, "nota": 1}}
            ]
        }},
        {"$project": {
            "max": {"$arrayElemAt": ["$max", 0]},
            "min": {"$arrayElemAt": ["$min", 0]}
        }}
    ]
list(alu.aggregate(pipeline))

[{'max': {'nombre': 'Marta', 'nota': 9, 'ciclo': 'DAW'},
  'min': {'nombre': 'Luis', 'nota': 5, 'ciclo': 'DAW'}}]

### join  -> no recomendables

In [74]:
cl = db["ciclos"] #coleccion

cl.insert_many([
    {"ciclo": "ASIR", "descripcion": "Adm. de sistemas Inf. en red"},
    {"ciclo": "DAW", "descripcion": "Desarrollo apl. web"},
    {"ciclo": "DAM", "descripcion": "Desarrollo apl. multiplataforma"},
    {"ciclo": "MAK", "descripcion": "Marketing"},
   
])

InsertManyResult([ObjectId('6989fcd6788be09fc21060f2'), ObjectId('6989fcd6788be09fc21060f3'), ObjectId('6989fcd6788be09fc21060f4'), ObjectId('6989fcd6788be09fc21060f5')], acknowledged=True)

In [75]:
#$lookup devuelve info_ciclo como array.Es un array de documentos con los registros coincidentes que encontró en la colección ciclos.
#$unwind lo “aplana” para tener un documento por alumno (y con preserveNull... no pierdes alumnos si no hay match).
pipeline = [
    {
        "$lookup": {
            "from": "ciclos",          # colección destino
            "localField": "ciclo",     # campo en alumnos
            "foreignField": "ciclo",   # campo en ciclos
            "as": "info_ciclo"         # array con el match
        }
    },
    {"$unwind": {"path": "$info_ciclo", "preserveNullAndEmptyArrays": True}},
    {
        "$project": {
            "_id": 0,
            "nombre": 1,
            "nota": 1,
            "ciclo": 1,
            "descripcion": "$info_ciclo.descripcion"
        }
    },
    {"$sort": {"ciclo": 1, "nombre": 1}}
]

list(alu.aggregate(pipeline))


[{'ciclo': 'ASIR', 'descripcion': 'Adm. de sistemas Inf. en red'},
 {'ciclo': 'ASIR', 'descripcion': 'Adm. de sistemas Inf. en red'},
 {'nombre': 'Ana',
  'nota': 8,
  'ciclo': 'ASIR',
  'descripcion': 'Adm. de sistemas Inf. en red'},
 {'nombre': 'Ana',
  'nota': 8,
  'ciclo': 'ASIR',
  'descripcion': 'Adm. de sistemas Inf. en red'},
 {'nombre': 'Paola',
  'nota': 9,
  'ciclo': 'DAM',
  'descripcion': 'Desarrollo apl. multiplataforma'},
 {'ciclo': 'DAN', 'descripcion': 'Desarrollo apl. multiplataforma'},
 {'ciclo': 'DAW', 'descripcion': 'Desarrollo apl. web'},
 {'ciclo': 'DAW', 'descripcion': 'Desarrollo apl. web'},
 {'nombre': 'David',
  'nota': 5,
  'ciclo': 'DAW',
  'descripcion': 'Desarrollo apl. web'},
 {'nombre': 'David',
  'nota': 5,
  'ciclo': 'DAW',
  'descripcion': 'Desarrollo apl. web'},
 {'nombre': 'Luis',
  'nota': 5,
  'ciclo': 'DAW',
  'descripcion': 'Desarrollo apl. web'},
 {'nombre': 'Luis',
  'nota': 5,
  'ciclo': 'DAW',
  'descripcion': 'Desarrollo apl. web'},
 {'nom